# Seguir un mineral de cobre por trituración y molienda

**Pregunta:** ¿Cómo cambian el tamaño alcanzado y la demanda de molienda al modificar el objetivo de un circuito cerrado?

Nivel: 1. Requisitos previos: Ninguno; comienza aquí. Tiempo de lectura: 18 minutos.



## Objetivos

Seguir el tamaño de alimentación a través del circuito de conminución a partir de un mineral OreFlow declarado y su punto operativo nominal. Distinguir un registro simulado de planta de una medición e identificar las magnitudes que calcula el motor.

## Preparación

```bash
pip install numpy==2.2.6 scipy==1.17.1 pandas==2.2.3 matplotlib==3.11.2 nbclient==0.11.0 nbformat==5.11.1 ipykernel==7.4.0 SALib==1.6.0 scikit-learn==1.5.2 ruff==0.15.18 pytest==9.1.1
```

El pipeline OreFlow se importa desde `data-pipeline/`; este notebook nunca instala paquetes.

Para la lección de nivel 1 sobre 01-comminution, registra las versiones fijadas antes de ejecutar el análisis; así se puede reconstruir el entorno que produjo cada tabla y figura.

In [ ]:
# Importar herramientas de análisis y registrar sus versiones exactas.
import importlib.metadata
import json
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

repo = Path("../../../").resolve()
sys.path.insert(0, str(repo / "data-pipeline"))
for package in ("numpy", "scipy", "pandas", "matplotlib", "nbclient", "nbformat", "ipykernel"):
    print(package, importlib.metadata.version(package))
legend_style = dict(loc="lower center", bbox_to_anchor=(0.5, 1.02), frameon=False)


## Datos

Responsable: autores de OreFlow, licencia MIT. El catálogo aporta mineral, triturador, molino, clasificador y operación nominal del pórfido blando. Las constantes y extremos de referencia provienen de engine/data/constants.json. Salidas y distribuciones de tamaño se calculan aquí, sin cargar una traza guardada. No se introduce distribución medida, campaña energética ni otro conjunto externo. Fuente: https://github.com/fsantibanezleal/CAOS_OreFlow.

## Método

El motor representa el tamaño en una malla logarítmica fija. La trituración transforma la distribución mediante el modelo de clasificación Whiten. La molienda aplica una función de selección específica por energía y una matriz de fractura; el clasificador envía cada clase de tamaño al rebose o al reciclaje. En estado estacionario, la alimentación fresca coincide con los productos finales dentro de la tolerancia del balance unitario. Aquí llamamos a `simulate` del pipeline OreFlow; no reimplementamos ecuaciones de conminución.

Importar el catálogo de casos OreFlow y el simulador del pipeline del repositorio. El punto nominal es el estado operativo documentado del caso; el solver devuelve el mismo resultado estructurado que usa el proceso de bake.

In [ ]:
# Importar el catálogo de casos OreFlow y el simulador del pipeline del repositorio.
from pipeline.cases.catalog import CASES
from pipeline.engine.circuit import simulate

case = next((item for item in CASES if item.id == "copper_porphyry_soft"))
result = simulate(case.ore, case.plant, case.nominal)
print("familia:", case.plant.family)
print("tratamiento:", case.nominal.throughput_tph, "t/h")
print("F80 de alimentación:", round(result.metrics["crusher_feed_f80_um"], 1), "um")
print("P80 de producto:", round(result.grinding.p80_um, 1), "um")
print("energía de molienda:", round(result.grinding.specific_energy_kwh_t, 2), "kWh/t")


Inspeccionar las distribuciones acumuladas de pasante en los límites identificados del circuito. P80 es un percentil de una distribución; no se infiere de una partícula representativa.

In [ ]:
# Inspeccionar las distribuciones acumuladas de pasante en los límites identificados del circuito.
plt.rcParams.update({"font.size": 18, "axes.titlepad": 40})
size = np.asarray(result.curves["size_um"])
psd = result.curves["psd"]
fig, ax = plt.subplots(figsize=(6.4, 3.8))
for stream in ("crusher_feed", "new_feed", "cyclone_overflow"):
    ax.plot(
        size,
        100.0 * np.asarray(psd[stream]),
        label={
            "crusher_feed": "Alimentación al triturador",
            "new_feed": "Alimentación fresca",
            "cyclone_overflow": "Rebose del ciclón",
        }.get(stream, stream),
    )
ax.set_xscale("log")
ax.set_xlim(max(1.0, float(size.min())), float(size.max()))
ax.set_ylim(0, 100)
ax.set(
    xlabel="Tamaño de partícula\n(µm)",
    ylabel="Pasante\nacumulado (%)",
    title="Distribuciones nominales",
)
ax.grid(True, alpha=0.25)
ax.legend(["Triturador", "Fresca", "Rebose"], ncol=3, **legend_style)
fig.subplots_adjust(left=0.22, right=0.96, bottom=0.23, top=0.72)
ax.locator_params(axis="x", numticks=4)
ax.tick_params(axis="x", which="minor", labelbottom=False)


Las curvas acumuladas del motor son fracciones. Multiplicar por 100 permite interpretar el 80% sobre la escala porcentual indicada. El dominio abarca la malla real e incluye el cruce F80 de alimentación al triturador, sin recortarlo por debajo de ese percentil. La separación de curvas describe reducción de tamaño en un circuito balanceado; no establece calibración mineral medida.

Leer las métricas calculadas por etapa y el residuo máximo del balance. Un error de cierre pequeño verifica conservación numérica en esta corrida; no valida el modelo frente a una planta.

In [ ]:
# Leer las métricas calculadas por etapa y el residuo máximo del balance.
from IPython.display import display
import pandas as pd

metrics = result.metrics
rows = [
    ("F80 de alimentación a trituración", metrics["crusher_feed_f80_um"], "µm"),
    ("P80 de trituración", metrics["crusher_p80_um"], "µm"),
    ("P80 de producto del circuito", metrics["p80_um"], "µm"),
    ("Energía específica de molienda", metrics["specific_energy_grinding_kwh_t"], "kWh/t"),
    ("Error relativo máximo de balance", result.balance["max_relative_error"], "fraction"),
]
display(pd.DataFrame(rows, columns=["Magnitud", "Valor", "Unidad"]))


Conectar las curvas anteriores con un extremo real de alimentación al molino. Resolver nuevamente el pórfido blando y tomar el F80 alcanzado desde la molienda. La alimentación al triturador es otro flujo; usar su F80 más grueso exageraría la reducción del molino. Conservar el índice en kWh/t y los tamaños en micrómetros. Así es posible comparar requerimiento de energía y circuito resuelto sin adivinar a qué flujo pertenece cada extremo.

In [ ]:
# Resolver el circuito nominal antes de evaluar hipótesis alternativas de energía.
from pipeline.cases.catalog import CASE_BY_ID
from pipeline.engine.circuit import simulate
from pipeline.engine.energy import law_curves

case = CASE_BY_ID["copper_porphyry_soft"]
nominal = simulate(case.ore, case.plant, case.nominal)
f80 = nominal.grinding.feed_f80_um
wi = case.nominal.work_index_kwh_t
print("F80 alcanzado del molino (µm):", round(f80, 2))


Evaluar Bond, Rittinger y Kick al producto nominal alcanzado. Las columnas son estimaciones alternativas en kWh por tonelada, no tres partes de la cuenta energética. Bond usa raíz cuadrada inversa del tamaño, Rittinger tamaño inverso y Kick logaritmo de la razón de tamaños. Sus diferencias importan incluso en un punto, antes de que un barrido revele las formas. Conservar los valores junto a la energía de molienda calculada independientemente, sin elegir solo el que parezca más cercano.

In [ ]:
# Calcular el trabajo calibrado al tamaño de producto alcanzado.
at_product = law_curves(wi, f80, np.array([nominal.grinding.p80_um]))
display(pd.DataFrame(at_product).round(3))


Solicitar ahora un producto veinte por ciento más fino, sin cambiar mineral, agua ni alimentación fresca. El circuito debe resolver su recirculación en lugar de sustituir el P80 impreso. Leer por separado tamaño solicitado y alcanzado: la potencia instalada puede impedir llegar al objetivo. La potencia requerida responde si la solicitud es realizable en esta configuración de autoría propia. Un objetivo es una entrada a un cálculo interconectado; no garantiza una salida ni constituye una instrucción para una planta real.

In [ ]:
# Solicitar al motor un producto más fino manteniendo fijos alimentación fresca y mineral.
fine_point = case.nominal.with_values(target_p80_um=case.nominal.target_p80_um * 0.8)
fine = simulate(case.ore, case.plant, fine_point)
print("P80 solicitado (µm):", fine_point.target_p80_um)
print("P80 alcanzado (µm):", round(fine.grinding.p80_um, 3))
print("Potencia requerida (kW):", round(fine.grinding.required_power_kw, 3))


## Resultados

**Respuesta:** Solicitar <!-- result:P80 fino solicitado (µm) -->0<!-- /result --> µm produce molienda alcanzada <!-- result:P80 fino alcanzado (µm) -->0<!-- /result --> µm frente a <!-- result:P80 nominal (µm) -->0<!-- /result --> µm nominales. La solicitud fina requiere <!-- result:Potencia fina requerida (kW) -->0<!-- /result --> kW. Es el circuito resuelto con recirculación, y no la entrada solicitada por sí sola, el que determina alcanzar ese producto.

Resumir molienda nominal y fina alcanzadas y potencia requerida desde el circuito resuelto para responder sobre el objetivo.

In [ ]:
# Resumir las salidas calculadas que sostienen la interpretación de esta lección.
answer_values = {
    "P80 nominal (µm)": nominal.grinding.p80_um,
    "P80 fino solicitado (µm)": fine_point.target_p80_um,
    "P80 fino alcanzado (µm)": fine.grinding.p80_um,
    "Potencia fina requerida (kW)": fine.grinding.required_power_kw,
    "Trabajo Bond (kWh/t)": at_product["bond"][0],
    "Trabajo Rittinger (kWh/t)": at_product["rittinger"][0],
    "Trabajo Kick (kWh/t)": at_product["kick"][0],
}
answer_values = {label: round(float(number), 3) for label, number in answer_values.items()}
print(json.dumps(answer_values, ensure_ascii=False))


## Resumen

La resolución utiliza el mismo solver de circuito que OreFlow y permite seguir una distribución de tamaños y comprobar su conservación. El cierre del modelo verifica consistencia interna; la validación frente a datos medidos de planta requiere evidencia independiente.

## Referencias

- https://doi.org/10.1016/0301-7516(80)90034-4
- https://github.com/fsantibanezleal/CAOS_OreFlow/tree/develop/data/derived
- https://doi.org/10.1016/0301-7516(80)90034-4